<a href="https://colab.research.google.com/github/ivan-penta/proyecto_cidam/blob/main/proyecto_cidam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Creción del ámbiente en PySpark

In [2]:
# 1. Instalar Java 17 y PySpark
!apt-get update -qq
!apt-get install openjdk-17-jdk -y
!pip install pyspark==4.0.1 -q

# 2. Configurar variables de entorno
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# 3. Verificar instalación de Java
!java -version

# 4. Crear SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis Compranet Big Data") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

print("✅ SparkSession creada")
print("Versión:", spark.version)
print("App Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-17-jdk is already the newest version (17.0.18+8-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 117 not upgraded.
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
✅ SparkSession creada
Versión: 4.0.1
App Name: Analisis Compranet Big Data
Master: local[*]


### Cargar dataset

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr

spark = SparkSession.builder \
    .appName("Analisis Compranet Big Data") \
    .getOrCreate()

print("📂 Cargando dataset completo de CompraNet...")


data = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .csv("compranet_historico.csv")

data = data.withColumn(
    "importe",
    expr("try_cast(importe as double)")
)

conteo_nulos = data.filter(col("importe").isNull()).count()
print(f"⚠️ Registros con formato inválido convertidos a NULL: {conteo_nulos}")

total_registros = data.count()
print(f"📊 Total de registros procesados: {total_registros:,}")

data.write.mode("overwrite").parquet("compranet_historico.parquet")
print("💾 Dataset completo guardado exitosamente en formato Parquet.")

📂 Cargando dataset completo de CompraNet...
⚠️ Registros con formato inválido convertidos a NULL: 5188
📊 Total de registros procesados: 2,356,612
💾 Dataset completo guardado exitosamente en formato Parquet.


In [5]:
from pyspark.sql.functions import col

registros_antes=data.count()

data = data.dropna(subset=[
    "importe","ff_fecha_inicio",
    "ff_fecha_fin"
])

registros_despues = data.count()

print(f"Registros antes de limpieza: {registros_antes}")
print(f"Registros después de limpieza: {registros_despues}")
print(f"Número de registros eliminados: {registros_antes - registros_despues}")

Registros antes de limpieza: 2356612
Registros después de limpieza: 2347566
Número de registros eliminados: 9046


In [6]:
num_registros =data.count()
num_columnas = len(data.columns)

print(f"Número de registros: {num_registros}")
print(f"Número de columnas: {num_columnas}")

Número de registros: 2347566
Número de columnas: 16


### Selección de variables relevantes

In [7]:
from pyspark.sql.functions import col

df = data.select(
    "codigo_contrato",
    "codigo_expediente",
    "proveedor",
    "contract_type",
    "tipo_contratacion",
    "tipo_expediente",
    "importe",
    "moneda",
    "fecha_inicio",
    "fecha_fin"
)

df.show(5)

+---------------+-----------------+--------------------+---------------+--------------------+--------------------+----------+------+--------------------+--------------------+
|codigo_contrato|codigo_expediente|           proveedor|  contract_type|   tipo_contratacion|     tipo_expediente|   importe|moneda|        fecha_inicio|           fecha_fin|
+---------------+-----------------+--------------------+---------------+--------------------+--------------------+----------+------+--------------------+--------------------+
|        2376191|          2161394|Equity Appraisal ...|    3.Servicios|           Servicios|05. Adjudicación ...|   89012.0|   MXN|2020-07-22 05:00:...|2020-08-27 04:59:...|
|             89|              348|Si Vale Mexico Sa...|ADQUISICIONES_0|No especificado p...|V20151220 12. Adj...| 5980292.7|   MXN|2010-12-06 06:00:...|2011-01-01 05:59:...|
|           1756|              399|Metlife Mexico Sa...|    SERVICIOS_1|No especificado p...|V20110525 01. Lic...| 3904647.0|

### Converción de fechas

In [8]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "fecha_inicio",
    to_timestamp(col("fecha_inicio"))
)

df = df.withColumn(
    "fecha_fin",
    to_timestamp(col("fecha_fin"))
)

In [9]:
df = df.filter(col("fecha_inicio").isNotNull())

#### Crear variable año

In [10]:
from pyspark.sql.functions import year

df = df.withColumn(
    "anio",
    year(col("fecha_inicio"))
)

df.select("fecha_inicio","anio").show(5)

+-------------------+----+
|       fecha_inicio|anio|
+-------------------+----+
|2020-07-22 05:00:00|2020|
|2010-12-06 06:00:00|2010|
|2011-01-01 06:00:00|2011|
|2011-01-01 06:00:00|2011|
|2011-01-01 06:00:00|2011|
+-------------------+----+
only showing top 5 rows


### Seleccionar periodo de estudio

In [11]:
df = df.filter(
    (col("anio") >= 2012) & (col("anio") <= 2024)
)
df = df.filter(df.fecha_inicio.isNotNull())

df.count()

2213776

### Variable sexenio

In [12]:
from pyspark.sql.functions import col, year, when

df = df.withColumn(
    "sexenio",
    when((col("anio") >= 2012) & (col("anio") <= 2018), "EPN")
    .when((col("anio") >= 2019) & (col("anio") <= 2024), "AMLO")
)

df.select("anio","sexenio").show(20)

+----+-------+
|anio|sexenio|
+----+-------+
|2020|   AMLO|
|2020|   AMLO|
|2020|   AMLO|
|2020|   AMLO|
|2020|   AMLO|
|2020|   AMLO|
|2020|   AMLO|
|2021|   AMLO|
|2021|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
|2022|   AMLO|
+----+-------+
only showing top 20 rows


### Monto total por sexenio

In [13]:
contratos_sexenio = df.groupBy("sexenio").count()

contratos_sexenio.show()

+-------+-------+
|sexenio|  count|
+-------+-------+
|    EPN|1438494|
|   AMLO| 775282|
+-------+-------+



### Monto total por sexenio

In [14]:
from pyspark.sql.functions import sum, format_number

monto_sexenio = df.groupBy("sexenio").agg(
    sum("importe").alias("monto_total")
)

monto_sexenio = monto_sexenio.withColumn(
    "monto_total",
    format_number("monto_total", 2)
)

monto_sexenio.select("sexenio", "monto_total").show(truncate=False)

+-------+--------------------+
|sexenio|monto_total         |
+-------+--------------------+
|EPN    |3,201,596,488,429.85|
|AMLO   |2,315,314,620,485.94|
+-------+--------------------+



### Importe promedio por contrato

In [15]:
from pyspark.sql.functions import avg
prom_sexenio = df.groupBy("sexenio").agg(
    avg("importe").alias("importe_promedio"))

prom_sexenio = prom_sexenio.withColumn(
    "importe_promedio",
    format_number("importe_promedio", 2)
)

prom_sexenio.select("sexenio", "importe_promedio").show(truncate=False)

+-------+----------------+
|sexenio|importe_promedio|
+-------+----------------+
|EPN    |2,225,658.56    |
|AMLO   |2,986,416.07    |
+-------+----------------+



### Modalidades de contratación por sexenio

In [17]:
modalidades = df.groupBy(
    "sexenio",
    "tipo_contratacion"
).count()

modalidades.orderBy("sexenio").show(50)

+-------+--------------------+------+
|sexenio|   tipo_contratacion| count|
+-------+--------------------+------+
|   AMLO|       Adquisiciones|473193|
|   AMLO|      Arrendamientos|  4382|
|   AMLO|Servicios Relacio...|  9490|
|   AMLO|        Obra Pública| 34030|
|   AMLO|            Sin dato| 13080|
|   AMLO|           Servicios|241107|
|    EPN|            Sin dato|  4326|
|    EPN|       Adquisiciones|737516|
|    EPN|           Servicios|490157|
|    EPN|        Obra Pública|166376|
|    EPN|      Arrendamientos|  8825|
|    EPN|Servicios Relacio...| 31087|
|    EPN|No especificado p...|   207|
+-------+--------------------+------+



### Porcentaje de cada modalidad

In [24]:
total_sexenio = total_sexenio.withColumnRenamed("count", "total_count")

from pyspark.sql.functions import round

modalidades_pct = modalidades.join(total_sexenio, "sexenio") \
    .withColumn(
        "porcentaje",
        round(col("count") / col("total_count") * 100, 2)
    )

modalidades_pct.show()

+-------+--------------------+------+-----------+----------+
|sexenio|   tipo_contratacion| count|total_count|porcentaje|
+-------+--------------------+------+-----------+----------+
|   AMLO|       Adquisiciones|473193|     775282|     61.03|
|    EPN|            Sin dato|  4326|    1438494|       0.3|
|   AMLO|      Arrendamientos|  4382|     775282|      0.57|
|   AMLO|Servicios Relacio...|  9490|     775282|      1.22|
|    EPN|       Adquisiciones|737516|    1438494|     51.27|
|    EPN|           Servicios|490157|    1438494|     34.07|
|    EPN|        Obra Pública|166376|    1438494|     11.57|
|   AMLO|        Obra Pública| 34030|     775282|      4.39|
|   AMLO|            Sin dato| 13080|     775282|      1.69|
|    EPN|      Arrendamientos|  8825|    1438494|      0.61|
|    EPN|Servicios Relacio...| 31087|    1438494|      2.16|
|    EPN|No especificado p...|   207|    1438494|      0.01|
|   AMLO|           Servicios|241107|     775282|      31.1|
+-------+---------------

### Proveedores por sexenio

In [31]:
from pyspark.sql.functions import sum, col, format_number

# Agrupar por sexenio y proveedor, sumando el importe
proveedores_monto = df.groupBy("sexenio", "proveedor").agg(
    sum("importe").alias("monto_total")
)

proveedores_monto.orderBy(col("monto_total").desc()).select(
    "sexenio",
    "proveedor",
    format_number(col("monto_total"), 2).alias("monto")
).show(20, truncate=False)

+-------+--------------------------------------------------------------------+------------------+
|sexenio|proveedor                                                           |monto             |
+-------+--------------------------------------------------------------------+------------------+
|EPN    |Grupo Farmacos Especializados Sa de Cv                              |104,159,181,443.22|
|EPN    |Operadora Cicsa, S.A. de C.V.                                       |92,194,153,302.93 |
|AMLO   |Electromecanica de Montacargas Sa de Cv                             |64,558,297,189.50 |
|AMLO   |Toka Internacional S a P I de Cv                                    |57,112,085,062.42 |
|EPN    |Farmaceuticos Maypo, S.A. de C.V.                                   |48,544,866,462.76 |
|EPN    |Distribuidora Internacional de Medicamentos y Equipo Medico Sa de Cv|39,471,156,925.61 |
|AMLO   |Alstom Transport Mexico Sa de Cv                                    |31,520,457,949.24 |
|EPN    |Grupo Const

### Top proveedores por sexenio

In [34]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, format_number

window = Window.partitionBy("sexenio").orderBy(col("monto_total").desc())

top_prov = proveedores_monto.withColumn(
    "rank",
    row_number().over(window)
)

top_prov.filter(col("rank") <= 10).select(
    "sexenio",
    "proveedor",
    format_number(col("monto_total"), 2).alias("monto_total"),
    "rank"
).show(truncate=False)

+-------+--------------------------------------------------------------------+------------------+----+
|sexenio|proveedor                                                           |monto_total       |rank|
+-------+--------------------------------------------------------------------+------------------+----+
|AMLO   |Electromecanica de Montacargas Sa de Cv                             |64,558,297,189.50 |1   |
|AMLO   |Toka Internacional S a P I de Cv                                    |57,112,085,062.42 |2   |
|AMLO   |Alstom Transport Mexico Sa de Cv                                    |31,520,457,949.24 |3   |
|AMLO   |Ica Constructora Sa de Cv                                           |27,851,444,297.41 |4   |
|AMLO   |Currie & Brown - Mexico Sa de Cv                                    |25,789,000,000.00 |5   |
|AMLO   |Farmaceuticos Maypo, S.A. de C.V.                                   |22,930,365,944.07 |6   |
|AMLO   |Laboratorios de Biologicos y Reactivos de Mexico Sa de Cv       

### Duración promedio de contratos

In [36]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    "duración_días",
    datediff(col("fecha_fin"), col("fecha_inicio")
))

dura_sexenio = df.groupBy("sexenio").agg(
    round(avg("duración_días"), 2).alias("duración_promedio")
)

dura_sexenio.show()

+-------+-----------------+
|sexenio|duración_promedio|
+-------+-----------------+
|    EPN|           114.57|
|   AMLO|           122.94|
+-------+-----------------+



### Evolución de contratos por año

In [37]:
contrato_anio = df.groupBy("sexenio","anio").count().orderBy("anio")

contrato_anio.show()

+-------+----+------+
|sexenio|anio| count|
+-------+----+------+
|    EPN|2012|170783|
|    EPN|2013|181812|
|    EPN|2014|197294|
|    EPN|2015|222760|
|    EPN|2016|233587|
|    EPN|2017|234168|
|    EPN|2018|198090|
|   AMLO|2019|193798|
|   AMLO|2020|162073|
|   AMLO|2021|203393|
|   AMLO|2022|197385|
|   AMLO|2023| 16798|
|   AMLO|2024|  1835|
+-------+----+------+

